# Semantic Model Memory Analysis
Companion to the inventory and BPA notebooks. This locally authored notebook calls Microsoft's Semantic Link Labs VertiPaq Analyzer API; it is not an official Microsoft notebook.

**Purpose:** Inspect available model/table/column storage statistics and find large or high-cardinality objects. This is not a measurement of report visual timings, peak query memory, or total Fabric capacity memory. Interpret results according to Import, Direct Lake, or DirectQuery storage mode; unavailable statistics are not zero usage, and DirectQuery source storage is not VertiPaq memory.

**Requirements:** Microsoft Fabric notebook, `semantic-link-labs`, an authorized target workspace/model, and appropriate capacity/XMLA access and model permissions. If the package is missing, run `%pip install semantic-link-labs` in a separate Fabric cell or use your approved Fabric Environment/package version.

**Safety:** The analysis is explicitly opt-in because it consumes capacity. No model edits, refreshes, or exports are requested. Additional data-derived statistics are independently opt-in and may scan data or affect model residency/cache. Results remain in notebook memory/output; review metadata before sharing.

[Microsoft example](https://github.com/microsoft/semantic-link-labs/wiki/Code-Examples#vertipaq-analyzer) | [API reference](https://semantic-link-labs.readthedocs.io/en/stable/sempy_labs.html#sempy_labs.vertipaq_analyzer)


In [ ]:
WORKSPACE_NAME = ""
MODEL_NAME = ""
RUN_MEMORY_ANALYSIS = False
READ_STATS_FROM_DATA = False

if not WORKSPACE_NAME.strip() or not MODEL_NAME.strip():
    raise ValueError("Set WORKSPACE_NAME and MODEL_NAME to the exact Fabric workspace and semantic model names.")


In [ ]:
memory_stats = None
if RUN_MEMORY_ANALYSIS:
    import sempy_labs as labs
    from IPython.display import display

    memory_stats = labs.vertipaq_analyzer(
        dataset=MODEL_NAME,
        workspace=WORKSPACE_NAME,
        export=None,
        read_stats_from_data=READ_STATS_FROM_DATA,
    )
    for name, df in memory_stats.items():
        print(f"{name}: {len(df)} records; showing at most 50. Full results are in memory_stats.")
        display(df.head(50))
else:
    print("Memory analysis skipped. Set RUN_MEMORY_ANALYSIS=True after approving capacity usage.")


## Interpret the statistics
Inspect each returned section and its units. Prioritize large column/dictionary sizes and high cardinality, then verify whether the columns are needed and how measures and relationships use them. Do not remove columns or change relationships based only on size. Full section DataFrames are accessible with `memory_stats[section_name]` after a successful run.

If relevant statistics are absent for your storage mode, consult the API documentation before considering `READ_STATS_FROM_DATA=True`; that setting is not a guarantee of complete memory accounting. Use Fabric capacity monitoring for capacity-level usage and Power BI Performance Analyzer for visual/DAX timings.

Errors propagate rather than being presented as empty or zero memory. No live analysis was run when this notebook was created. All optimization changes require separate review.
